# SqlWorldCup

[Codility task](https://app.codility.com/programmers/trainings/6/sql_world_cup/) - Exercise 6, SQL - *medium*

Tables `teams(team_id, team_name)` and `matches(match_id, host_team, guest_team, host_goals, guest_goals)`.

Scoring: win = 3 points, draw = 1, loss = 0. Return **every** team with its `num_points`, ordered by `num_points` descending and then `team_id` ascending.

The SQL lives in `solution.sql` (the file you paste into Codility). This notebook loads `setup.sql` into an in-memory **SQLite** database, runs the solution and checks it.

In [ ]:
import sqlite3
from pathlib import Path

HERE = Path.cwd()          # Jupyter starts the kernel in the notebook's folder


def fresh_db(with_example=True):
    """New in-memory SQLite database; optionally loaded with setup.sql (tables + example data)."""
    conn = sqlite3.connect(":memory:")
    if with_example:
        conn.executescript((HERE / "setup.sql").read_text(encoding="utf-8"))
    return conn


def run(conn, sql_file="solution.sql"):
    """Execute the single query stored in `sql_file` and return all its rows."""
    return conn.execute((HERE / sql_file).read_text(encoding="utf-8")).fetchall()


def show(rows, headers):
    """Tiny table printer (no pandas needed)."""
    table = [headers] + [[str(c) for c in row] for row in rows]
    widths = [max(len(r[i]) for r in table) for i in range(len(headers))]
    for n, row in enumerate(table):
        print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))
        if n == 0:
            print("-+-".join("-" * w for w in widths))

## The solution

In [ ]:
print((HERE / "solution.sql").read_text(encoding="utf-8"))

## How it works

1. Each match contributes **two** rows `(team_id, points)`: the host's and the guest's, each computed with `case when goals > opponent_goals then 3 when goals = opponent_goals then 1 else 0 end`. They are stacked with `union all`.
2. We start from `teams` and `left join` those rows, so a team without matches still appears.
3. `coalesce(sum(points), 0)` turns the `NULL` of a team without matches into `0`.
4. `order by num_points desc, team_id asc`.

## Official example

In [ ]:
conn = fresh_db()
rows = run(conn)
show(rows, ["team_id", "team_name", "num_points"])
assert rows == [(20, "Never", 4), (50, "Gonna", 4), (10, "Give", 3), (30, "You", 3), (40, "Up", 0)]
print("\nexample OK")

## Edge cases

Each case creates its own teams and matches.

In [ ]:
def standings(teams, matches, sql_file="solution.sql"):
    """teams: [(id, name)], matches: [(match_id, host, guest, host_goals, guest_goals)]."""
    conn = fresh_db()
    conn.execute("delete from teams")
    conn.execute("delete from matches")
    conn.executemany("insert into teams (team_id, team_name) values (?, ?)", teams)
    conn.executemany(
        "insert into matches (match_id, host_team, guest_team, host_goals, guest_goals) values (?, ?, ?, ?, ?)",
        matches,
    )
    return run(conn, sql_file)


T = [(3, "C"), (1, "A"), (2, "B")]

# no matches: everybody has 0 points, ordered by team_id
assert standings(T, []) == [(1, "A", 0), (2, "B", 0), (3, "C", 0)]

# a draw gives 1 point to each side, including a 0-0 draw
assert standings(T, [(1, 1, 2, 0, 0), (2, 2, 3, 2, 2)]) == [(2, "B", 2), (1, "A", 1), (3, "C", 1)]

# a team that never played still appears with 0 (team C)
assert standings(T, [(1, 1, 2, 3, 0)]) == [(1, "A", 3), (2, "B", 0), (3, "C", 0)]

# guest wins are counted for the guest, not for the host
assert standings(T, [(1, 1, 2, 0, 1)]) == [(2, "B", 3), (1, "A", 0), (3, "C", 0)]

# ties on points are ordered by team_id ascending
assert [r[0] for r in standings(T, [(1, 1, 2, 1, 0), (2, 3, 2, 1, 0)])] == [1, 3, 2]
print("edge cases OK")

## Pitfall: `UNION` instead of `UNION ALL`

`UNION` removes duplicate rows. If a team wins two matches, both contribute the identical row `(team_id, 3)` and one of them disappears, so the team gets 3 points instead of 6. The cell below runs a deliberately wrong variant to show it.

In [ ]:
wrong_sql = (HERE / "solution.sql").read_text(encoding="utf-8").replace("union all", "union")
(HERE / "_wrong_union.sql").write_text(wrong_sql, encoding="utf-8")
try:
    # team 1 beats team 2 twice (matches 1 and 2): correct total is 6
    good = standings(T, [(1, 1, 2, 1, 0), (2, 1, 2, 2, 0)])
    bad = standings(T, [(1, 1, 2, 1, 0), (2, 1, 2, 2, 0)], sql_file="_wrong_union.sql")
finally:
    (HERE / "_wrong_union.sql").unlink()

print("UNION ALL:", good[0])
print("UNION    :", bad[0])
assert good[0] == (1, "A", 6)
assert bad[0] == (1, "A", 3)          # wrong: the two identical rows collapsed

## Randomised check against a Python reference

In [ ]:
import random


def reference(teams, matches):
    points = {team_id: 0 for team_id, _ in teams}
    for _, host, guest, hg, gg in matches:
        if hg > gg:
            points[host] += 3
        elif hg == gg:
            points[host] += 1
            points[guest] += 1
        else:
            points[guest] += 3
    names = dict(teams)
    rows = [(tid, names[tid], pts) for tid, pts in points.items()]
    return sorted(rows, key=lambda r: (-r[2], r[0]))


rng = random.Random(8)
for _ in range(400):
    n = rng.randint(2, 8)
    ids = rng.sample(range(1, 100), n)
    teams = [(tid, f"team{tid}") for tid in ids]
    matches = []
    for match_id in range(1, rng.randint(0, 25) + 1):
        host, guest = rng.sample(ids, 2)
        matches.append((match_id, host, guest, rng.randint(0, 3), rng.randint(0, 3)))
    assert standings(teams, matches) == reference(teams, matches), (teams, matches)
print("SQL result equals the Python reference on 400 random tournaments")

## Notes

- The classic way to see a match "from both sides" is to unpivot it: one row per participant. `UNION ALL` is the idiom for that.
- Starting from `teams` with a `LEFT JOIN` (instead of from the matches) is what keeps the teams without matches in the ranking.
- Alternative: two separate aggregations (as host, as guest) joined to `teams`, adding the two totals with `coalesce`. More text, same result.
- Everything used (CTE, `CASE`, `UNION ALL`, `LEFT JOIN`, `COALESCE`) is standard SQL, so it also runs on PostgreSQL.